In [ ]:
import yfinance as yf
from pandas import Timestamp
from tqdm.notebook import tqdm
import pandas as pd
import os
import numpy as np
import seaborn as sns 
import pickle


In [ ]:
path=f"../graphsV2/COLLLevel-MAX"
SOGLIA=.5
SOGLIA_CORRELAZIONE=.0
seg="Monthly"

In [ ]:
def getTickers(tickers):
    series=None
    for ticker in tqdm(tickers):
        tk = yf.Ticker(ticker)
        if seg=="Weekly":
            prices=tk.history(period="4y",interval="1wk")
            prices["snapshot"]=[str((Timestamp(t).year,(Timestamp(t).week-1))) for t in prices.index.values]
        if "Monthly" in seg:
            prices=tk.history(period="5y",interval="1mo")
            prices["snapshot"]=[str((Timestamp(t).year,(Timestamp(t).month-1))) for t in prices.index.values]
        if seg=="Trimester":
            prices=tk.history(period="5y",interval="3mo")
            prices["snapshot"]=[str((Timestamp(t).year,(Timestamp(t).month-1)//3)) for t in prices.index.values]

        prices.index=prices["snapshot"]
        prices[ticker]=prices["Close"]
        if series is None:
            series=prices[[ticker]]
        else:
            series=series.join(prices[ticker])
        
    return series.drop_duplicates()

tickers=["BTC-USD","ETH-USD","GAS-ETH"]
prices=getTickers(tickers)
prices["GAS-USD"]=prices["GAS-ETH"]*prices["ETH-USD"]
tickers=list(prices.columns.values)


In [ ]:
s = sorted(os.listdir(path), key=lambda x: eval(x)[0]*100+eval(x)[1])


In [ ]:
x=[]
LENS=[]
for snap in tqdm(s):
    df=pd.read_csv(f"{path}/{snap}/metadata.csv")
    LENS.append((len(df),len(df.drop_duplicates(keep="first"))))
    x.append({
        "snapshot":snap,
        "sum(volume)":df["volume"].sum(),
        "mean(volume)":df["volume"].mean(),

        "mean(max(price))":df["max_price"].mean(),
        "mean(min(price))":df["min_price"].mean(),
        "mean(mean(price))":df["avg_price"].mean(),
        
        "sum(#tx)":df["#tx"].sum(),
        "max(#tx)":df["#tx"].max(),
        "min(#tx)":df["#tx"].min(),
        "mean(#tx)":df["#tx"].mean(),
        })
volumes=pd.DataFrame.from_records(x)
volumes.index=volumes.snapshot
assert sum([l[0]-l[1] for l in LENS])==0

In [ ]:
sim=[]

for snap in tqdm(s):
    try:
        nodes=pd.read_csv(f"{path}/{snap}/metadata.csv",index_col="id")
        nodes_nft=pd.read_csv(f"../graphsV2/NFTLevel-Pruning_V2/{snap}/metadata.csv",index_col=0)

        # aggiunta 
        nodes_nft['node'] = nodes_nft.index
        nodes_nft['collection'] = nodes_nft['node'].apply(lambda x: eval(x)[0])
        edges=pd.read_csv(f"{path}/{snap}/edge_list.csv").drop_duplicates(keep='last')
        edges=edges[edges.weight>SOGLIA]
        assert (edges["size_source"]>=edges["used_source"]).all()
        assert (edges["size_target"]>=edges["used_target"]).all()

        edges["category_src"]=edges["source"].apply(lambda x: nodes.loc[x]["category"])
        edges["category_dst"]=edges["target"].apply(lambda x: nodes.loc[x]["category"])


        sim.append({
                "snapshot":snap,
                "sum(sim)":edges["weight"].sum(),
                "mean(sim)":edges["weight"].mean(),
                "intra":edges[edges.category_src==edges.category_dst].weight.mean(),
                "inter":edges[edges.category_src!=edges.category_dst].weight.mean()
        })



    except Exception as e:
        print(e)
        sim.append({"snapshot":snap,
                "sum(sim)":0,
                "mean(sim)":0,
                "intra":0,
                "inter":0,
        })       
sim=pd.DataFrame().from_records(sim)
sim.index=sim.snapshot


In [ ]:
keys=set(prices.index.values)
keys.intersection_update(volumes.index.values)

keys.intersection_update(sim.index.values)

keys=list(set(keys))

keys=sorted(keys,key=lambda x: eval(x)[0]*52+eval(x)[1])

df=pd.concat([volumes.loc[keys],sim.loc[keys],prices.loc[keys]],axis=1)[list(volumes.columns.values)[1:]+list(sim.columns.values)[1:]+tickers]


In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 1, figsize=(21, 15))
df[["inter", "intra", "mean(sim)"]].plot(ax=ax[0])
df[["mean(volume)", "mean(min(price))", "mean(max(price))","mean(mean(price))"]
   ].plot(ax=ax[1])
df[tickers].plot(ax=ax[2], logy=True)


In [ ]:
# plt.figure(figsize=(14,12))
# df_corr=df.corr()
# df_corr=df_corr.where(np.abs(df_corr)>SOGLIA_CORRELAZIONE,0)
# sns.heatmap(df_corr, vmin=-1, vmax=1, annot=True,cmap=sns.color_palette("coolwarm", as_cmap=True))
# print("Matrice di correlazione tra le grandezze osservate")
# df_corr

In [ ]:
def crosscorr(datax, datay, lag=0, wrap=False):
    """ Lag-N cross correlation. 
    Shifted data filled with NaNs 
    
    Parameters
    ----------
    lag : int, default 0
    datax, datay : pandas.Series objects of equal length
    Returns
    ----------
    crosscorr : float
    """
    if wrap:
        shiftedy = datay.shift(lag)
        shiftedy.iloc[:lag] = datay.iloc[-lag:].values
        return datax.corr(shiftedy)
    else: 
        return datax.corr(datay.shift(lag))


In [ ]:
#ESEMPIO
# genero 2 timeseries sinusoidali sfasate di 3 timestep sin2 è "anticipata rispetto" a sin
#l'idea è capire se il primo massimo di correlazione lo si incontra a destra dello 0 nell'istante di sfasamento pari a 3
sin=np.sin(np.linspace(0,2*np.pi*5,100))
sin2=-sin[3:]
plt.plot(sin,label="sin")
plt.plot(sin2,label="sin2")
plt.legend()
plt.show()

x,y=[],[]
for lag in range(-12,12):
    x.append(lag)
    corr=crosscorr(pd.Series(sin), pd.Series(sin2), lag=lag)
    y.append(corr)
best_offset=x[np.argmax(np.abs(y))]
print(f"BEST OFFSET {best_offset}")
plt.plot(x,y)
plt.axvline(best_offset,color='black',linestyle='--',label='Center')
plt.show()

In [ ]:
CROSS_CORR={}
for lag in range(-12,12):
    curr_cross_corr={}
    for c1 in [
        "mean(max(price))",
        "mean(min(price))",
        "mean(mean(price))",
        "mean(volume)"
        ]:
        
        curr_cross_corr[c1]={}
        for c2 in [
            "mean(sim)",
            # "inter",
            # "intra",
            "BTC-USD"]:
            corr=crosscorr(df[c1], df[c2], lag=lag)
            curr_cross_corr[c1][c2]=corr
        CROSS_CORR[lag]=pd.DataFrame.from_dict(curr_cross_corr)

Con offset negativo il leader è la grandezza a sinistra, mentre il follower quella a destra,
altrimenti viceversa.

In [ ]:
import matplotlib
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 40}

matplotlib.rc('font', **font)
plt.rcParams['text.usetex'] = True
for idx in CROSS_CORR[0].index.values:
    fig,ax=plt.subplots(1,1,figsize=(21,12))
    datas=[]
    ax.set_title(f"$q_2=$ {idx}")
    for column in CROSS_CORR[0].columns.values:
        all_lags=list(CROSS_CORR.keys())
        all_corrs=[CROSS_CORR[lag][column][idx] for lag in all_lags]
        datas.append(np.array(all_corrs).reshape(1,-1))

    
    t_synch=np.argmax(np.mean(np.abs(datas),axis=0))
    ax.axvline(all_lags[t_synch],color='r',linestyle='--',label='Peak synchrony')
    # ax.scatter(all_lags[t_synch],datas[:,t_synch].mean(),color='r',s=600)
    ax.fill_between([x for x in all_lags if x>=0], -1.2,1.2,alpha=.1,color="green")
    ax.fill_between([x for x in all_lags if x<=0], -1.2,1.2,alpha=.1,color="red")

    for column in CROSS_CORR[0].columns.values:
        all_lags=list(CROSS_CORR.keys())
        all_corrs=[CROSS_CORR[lag][column][idx] for lag in all_lags]
        best_t=np.argmax(np.abs(all_corrs))
        best_corr=all_corrs[best_t]
        ax.plot(all_lags,all_corrs,label=f"$q_1=${column}")
    datas=np.concatenate(datas)
    ax.fill_between(all_lags, np.min(datas,0),np.max(datas,0),alpha=.1,color="orange")

    ax.set_xlabel("Offset")
    ax.set_ylabel("Correlation")
    ax.legend()
    ax.grid(linestyle="--")
    plt.savefig(f"../graphsV2/plots/COLL-Level-MEAN-TLCC-{idx}.png")

In [ ]:
import matplotlib
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 50}

matplotlib.rc('font', **font)
plt.rcParams['text.usetex'] = True
for idx in CROSS_CORR[0].index.values:
    fig,ax=plt.subplots(1,1,figsize=(21,12))
    datas=[]
    ax.set_title(f"$q_2=$ {idx}")
    for column in CROSS_CORR[0].columns.values:
        all_lags=list(CROSS_CORR.keys())
        all_corrs=[CROSS_CORR[lag][column][idx] for lag in all_lags]
        datas.append(np.array(all_corrs).reshape(1,-1))
    datas=np.concatenate(datas)
    
    t_synch=np.argmax(np.mean(np.abs(datas),axis=0))

    # ax.fill_between([x for x in all_lags if x>=0], -1,1,alpha=.1,color="green")
    # ax.fill_between([x for x in all_lags if x<=0], -1,1,alpha=.1,color="red")

    for column in CROSS_CORR[0].columns.values:
        all_lags=list(CROSS_CORR.keys())
        all_corrs=[CROSS_CORR[lag][column][idx] for lag in all_lags]
        t_synch=np.argmax(np.abs(all_corrs))
        best_corr=all_corrs[best_t]
        ax.plot(all_lags,all_corrs,label=f"$q_1=${column}",linewidth=4)
        ax.scatter(all_lags[t_synch],all_corrs[t_synch],color='green',s=600)
        print(f'T:{all_lags[t_synch]} CORR:{all_corrs[t_synch]:3f}')
    print()
    # print(f'T:{all_lags[t_synch]} CORR:{np.mean(datas,0)[t_synch]:3f}')
    # ax.plot(all_lags, np.mean(datas,0),color="orange", linestyle="--",linewidth=4,label=f"$q_1=${idx} $q_2=$ prices")
    # ax.plot(all_lags, np.max(datas,0),color="orange", linestyle=":",linewidth=2)
    # ax.plot(all_lags, np.min(datas,0),color="orange", linestyle=":",linewidth=2)
    # ax.fill_between(all_lags, np.min(datas,0),np.max(datas,0),alpha=.2,color="orange")




    # if all_lags[t_synch]>=0:
    #     ax.axvline(all_lags[t_synch],color='r',linestyle='--',label='Peak synchrony',linewidth=4)
    #     ax.axhline(np.mean(datas,0)[t_synch],color='r',linestyle='--',linewidth=4)
    #     ax.scatter(all_lags[t_synch],datas[:,t_synch].mean(),color='r',s=600)
    # else:
    #     ax.axvline(all_lags[t_synch],color='green',linestyle='--',label='Peak synchrony',linewidth=4)
    #     ax.axhline(np.mean(datas,0)[t_synch],color='green',linestyle='--',linewidth=4)
    #     ax.scatter(all_lags[t_synch],datas[:,t_synch].mean(),color='green',s=600)


    ax.set_xlabel("Offset")
    ax.set_ylabel("Correlation")
    ax.legend()
    ax.grid(linestyle="--")
    ax.set_ylim([-1.2,1.2])
    ax.set_xlim([-14,14])
    plt. margins(x=0,y=0)

    plt.savefig(f"../graphsV2/plots/COLL-Level-TLCC-{idx}.png")

In [ ]:
fig,axes=plt.subplots(CROSS_CORR[0].shape[0],CROSS_CORR[0].shape[1],figsize=(25,21))
axes=axes.flatten()
CORR_LIM=0.5
ax_idx=0
for idx in CROSS_CORR[0].index.values:
    for column in CROSS_CORR[0].columns.values:
        all_lags=list(CROSS_CORR.keys())
        all_corrs=[CROSS_CORR[lag][column][idx] for lag in all_lags]
        best_t=np.argmax(np.abs(all_corrs))
        best_corr=all_corrs[best_t]
        axes[ax_idx].set_title(f"{column} <> {idx} : {best_corr:.3f}@t={all_lags[best_t]}")
        axes[ax_idx].plot(all_lags,all_corrs)
        axes[ax_idx].fill_between(all_lags, [CORR_LIM for _ in all_lags],[-CORR_LIM for _ in all_lags],alpha=.1)
        axes[ax_idx].axhline(-CORR_LIM,color='blue',linestyle='--',alpha=.2)
        axes[ax_idx].axhline(CORR_LIM,color='blue',linestyle='--',alpha=.2,label="No corr. zone")
        if abs(best_corr)>CORR_LIM:
            axes[ax_idx].axvline(all_lags[np.argmax(np.abs(all_corrs))],color='r',linestyle='--',label='Peak synchrony')
        axes[ax_idx].axvline(0,color='black',linestyle='--',label='Center')
        axes[ax_idx].set_xlabel(seg+" Offset")
        axes[ax_idx].set_ylabel("Correlation")
        axes[ax_idx].set_ylim([-1,1])
        axes[ax_idx].legend()
        ax_idx+=1 